# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_12 — SABR Smile Calibration

### Research question

How can the SABR model be calibrated to an implied-volatility smile, and what do its parameters reveal about level, skew, curvature, and backbone risk?

This notebook follows naturally from:

```text
02_09_volatility_surface_construction
02_11_heston_model_calibration
```

Heston is a stochastic-volatility model with a full option-pricing characteristic function. SABR is often used more directly as a **smile parameterisation**, especially for rates, FX, commodity options, and futures options.

This notebook covers:

1. SABR forward dynamics;
2. the role of $\alpha,\beta,\rho,\nu$;
3. Hagan's lognormal implied-volatility approximation;
4. Black-76 option pricing from forward implied volatility;
5. synthetic SABR smile generation;
6. noisy bid/ask option-chain construction;
7. fixed-$\beta$ smile calibration by maturity;
8. objective design and parameter transforms;
9. model-vs-market IV residual diagnostics;
10. smile feature interpretation;
11. sensitivity to $\rho$, $\nu$, and $\beta$;
12. shifted SABR notes for low/negative forwards;
13. saved calibration reports and manifest.

The key idea:

> SABR calibration is a practical way to turn a noisy smile into interpretable parameters: level, skew, curvature, and backbone.

## 1. SABR dynamics

The SABR model describes a forward price $F_t$ and a stochastic volatility factor $\alpha_t$:

$$
dF_t = \alpha_t F_t^\beta dW_t^F
$$
$$
d\alpha_t = \nu \alpha_t dW_t^\alpha
$$
with correlation:

$$
dW_t^F dW_t^\alpha = \rho dt
$$
The parameters are:

| Parameter | Meaning |
|---|---|
| $\alpha$ | initial volatility level |
| $\beta$ | elasticity/backbone parameter |
| $\rho$ | correlation, drives skew |
| $\nu$ | vol-of-vol, drives smile curvature |

Typical practical workflow:

1. choose or fix $\beta$;
2. calibrate $\alpha,\rho,\nu$ for each maturity;
3. inspect parameter stability and residuals.

## 2. Why SABR after Heston?

Heston is a full stochastic-volatility pricing model.

SABR is often used as a smile model because:

- it is fast;
- it gives intuitive parameters;
- it works naturally with forwards;
- it is widely used in rates and commodity/futures options;
- it can fit skew and curvature slice by slice.

The cost is that SABR calibration is usually more of a **smile interpolation and extrapolation tool** than a complete cross-maturity dynamic model.

## 3. Hagan lognormal SABR approximation

For $F\neq K$, Hagan's lognormal SABR implied volatility approximation is:

$$
\begin{aligned}
\sigma_{SABR}(F,K,T) &= \frac{\alpha} {(FK)^{(1-\beta)/2} \Bigg[ 1+ \frac{(1-\beta)^2}{24}\log^2(F/K) \\
&\quad + \frac{(1-\beta)^4}{1920}\log^4(F/K) \Bigg]} \frac{z}{x(z)} \Bigg[ 1+ A T \Bigg]
\end{aligned}
$$
where:

$$
\begin{aligned}
z &= \frac{\nu}{\alpha} (FK)^{(1-\beta)/2} \log(F/K)
\end{aligned}
$$
$$
\begin{aligned}
x(z) &= \log \left( \frac{ \sqrt{1-2\rho z+z^2}+z-\rho } {1-\rho} \right)
\end{aligned}
$$
and:

$$
\begin{aligned}
A &= \frac{(1-\beta)^2}{24} \frac{\alpha^2}{(FK)^{1-\beta}} \\
&\quad + \frac{\rho\beta\nu\alpha}{4(FK)^{(1-\beta)/2}} \\
&\quad + \frac{2-3\rho^2}{24}\nu^2
\end{aligned}
$$
For $F=K$, we use the ATM limit.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import least_squares
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

SCIPY_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class SABRParams:
    alpha: float
    beta: float
    rho: float
    nu: float


@dataclass(frozen=True)
class SABRMarketConfig:
    spot: float = 100.0
    risk_free_rate: float = 0.035
    dividend_yield: float = 0.005
    seed: int = 42


market_config = SABRMarketConfig()

true_params_by_maturity = {
    30/365: SABRParams(alpha=0.34, beta=0.70, rho=-0.42, nu=0.55),
    90/365: SABRParams(alpha=0.31, beta=0.70, rho=-0.38, nu=0.48),
    180/365: SABRParams(alpha=0.29, beta=0.70, rho=-0.34, nu=0.42),
    365/365: SABRParams(alpha=0.27, beta=0.70, rho=-0.30, nu=0.36),
    730/365: SABRParams(alpha=0.25, beta=0.70, rho=-0.25, nu=0.30),
}

pd.DataFrame([
    {"maturity": T, **asdict(p)}
    for T, p in true_params_by_maturity.items()
])

## 5. Black-76 helpers

SABR is naturally expressed for forwards.

Given a forward $F$, strike $K$, discount factor $D$, maturity $T$, and implied volatility $\sigma$, the Black-76 call price is:

$$
C = D\left(FN(d_1)-KN(d_2)\right)
$$
where:

$$
\begin{aligned}
d_1 &= \frac{\log(F/K)+\frac{1}{2}\sigma^2T}{\sigma\sqrt T}
\end{aligned}
$$
$$
d_2=d_1-\sigma\sqrt T
$$

In [ ]:
def normal_cdf(x):
    x = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def forward_price(spot, maturity, r, q):
    return spot * exp((r - q) * maturity)


def discount_factor(maturity, r):
    return exp(-r * maturity)


def black76_price(forward, strike, maturity, discount, vol, option_type="call"):
    if maturity <= 0:
        if option_type == "call":
            return discount * max(forward - strike, 0.0)
        if option_type == "put":
            return discount * max(strike - forward, 0.0)
        raise ValueError("option_type must be call or put")

    if vol <= 0:
        if option_type == "call":
            return discount * max(forward - strike, 0.0)
        if option_type == "put":
            return discount * max(strike - forward, 0.0)

    d1 = (log(forward / strike) + 0.5 * vol**2 * maturity) / (vol * sqrt(maturity))
    d2 = d1 - vol * sqrt(maturity)

    if option_type == "call":
        return float(discount * (forward * normal_cdf(d1) - strike * normal_cdf(d2)))
    if option_type == "put":
        return float(discount * (strike * normal_cdf(-d2) - forward * normal_cdf(-d1)))
    raise ValueError("option_type must be call or put")


def black76_bounds(forward, strike, discount, option_type="call"):
    if option_type == "call":
        return discount * max(forward - strike, 0.0), discount * forward
    if option_type == "put":
        return discount * max(strike - forward, 0.0), discount * strike
    raise ValueError("option_type must be call or put")


def black76_implied_vol(price, forward, strike, maturity, discount, option_type="call", lo=1e-6, hi=4.0):
    lower, upper = black76_bounds(forward, strike, discount, option_type)
    if price < lower - 1e-10 or price > upper + 1e-10:
        return np.nan

    def objective(vol):
        return black76_price(forward, strike, maturity, discount, vol, option_type) - price

    f_lo = objective(lo)
    f_hi = objective(hi)

    if f_lo * f_hi > 0:
        return np.nan

    for _ in range(120):
        mid = 0.5 * (lo + hi)
        f_mid = objective(mid)
        if abs(f_mid) < 1e-10 or hi - lo < 1e-10:
            return float(mid)
        if f_lo * f_mid <= 0:
            hi = mid
            f_hi = f_mid
        else:
            lo = mid
            f_lo = f_mid

    return float(0.5 * (lo + hi))

## 6. Hagan lognormal SABR volatility function

We implement the lognormal implied-volatility approximation.

Important numerical cases:

- when $F\approx K$, use the ATM limit;
- when $z\approx0$, use $z/x(z)\approx1$;
- enforce positive $\alpha,\nu$ and $-1<\rho<1$.

In [ ]:
def sabr_lognormal_vol(forward, strike, maturity, params: SABRParams):
    F = float(forward)
    K = float(strike)
    T = float(maturity)

    alpha = float(params.alpha)
    beta = float(params.beta)
    rho = float(params.rho)
    nu = float(params.nu)

    if F <= 0 or K <= 0:
        return np.nan
    if alpha <= 0 or nu < 0 or beta < 0 or beta > 1 or abs(rho) >= 1:
        return np.nan

    one_minus_beta = 1.0 - beta
    log_fk = log(F / K)

    if abs(log_fk) < 1e-8:
        F_power = F ** one_minus_beta
        correction = (
            (one_minus_beta**2 / 24.0) * (alpha**2 / (F_power**2))
            + (rho * beta * nu * alpha) / (4.0 * F_power)
            + ((2.0 - 3.0 * rho**2) / 24.0) * nu**2
        )
        return float((alpha / F_power) * (1.0 + correction * T))

    FK_beta = (F * K) ** (0.5 * one_minus_beta)

    denominator = FK_beta * (
        1.0
        + (one_minus_beta**2 / 24.0) * log_fk**2
        + (one_minus_beta**4 / 1920.0) * log_fk**4
    )

    z = (nu / alpha) * FK_beta * log_fk

    sqrt_term = np.sqrt(1.0 - 2.0 * rho * z + z**2)
    x_z = np.log((sqrt_term + z - rho) / (1.0 - rho))

    z_over_x = 1.0 if abs(z) < 1e-8 or abs(x_z) < 1e-8 else z / x_z

    correction = (
        (one_minus_beta**2 / 24.0) * (alpha**2 / ((F * K) ** one_minus_beta))
        + (rho * beta * nu * alpha) / (4.0 * ((F * K) ** (0.5 * one_minus_beta)))
        + ((2.0 - 3.0 * rho**2) / 24.0) * nu**2
    )

    vol = (alpha / denominator) * z_over_x * (1.0 + correction * T)

    return float(max(vol, 1e-8))


# Quick sanity check.
F = forward_price(market_config.spot, 1.0, market_config.risk_free_rate, market_config.dividend_yield)
sabr_lognormal_vol(F, 100.0, 1.0, true_params_by_maturity[1.0])

## 7. Generate a synthetic SABR option chain

For each maturity:

1. compute the forward;
2. compute SABR implied volatility by strike;
3. price a Black-76 call;
4. add small bid/ask noise;
5. recover market IV from noisy midpoint.

This creates a realistic calibration dataset with known true parameters.

In [ ]:
def generate_synthetic_sabr_chain(params_by_maturity, market: SABRMarketConfig, strikes_by_maturity, seed=42):
    rng = np.random.default_rng(seed)
    rows = []

    for T, params in params_by_maturity.items():
        F = forward_price(market.spot, T, market.risk_free_rate, market.dividend_yield)
        D = discount_factor(T, market.risk_free_rate)

        for K in strikes_by_maturity[T]:
            true_iv = sabr_lognormal_vol(F, float(K), float(T), params)

            model_price = black76_price(
                forward=F,
                strike=float(K),
                maturity=float(T),
                discount=D,
                vol=true_iv,
                option_type="call"
            )

            spread = max(0.01, 0.002 * model_price + 0.02 * abs(np.log(K / F)))
            noise = rng.normal(0.0, 0.15 * spread)
            mid = max(model_price + noise, 1e-6)
            bid = max(mid - spread / 2, 0.0)
            ask = mid + spread / 2

            market_iv = black76_implied_vol(
                price=mid,
                forward=F,
                strike=float(K),
                maturity=float(T),
                discount=D,
                option_type="call"
            )

            rows.append({
                "maturity": float(T),
                "forward": F,
                "discount_factor": D,
                "strike": float(K),
                "log_moneyness": log(float(K) / F),
                "true_iv": true_iv,
                "model_price_true": model_price,
                "bid": bid,
                "ask": ask,
                "mid": mid,
                "bid_ask_spread": ask - bid,
                "market_iv": market_iv,
                "true_alpha": params.alpha,
                "true_beta": params.beta,
                "true_rho": params.rho,
                "true_nu": params.nu
            })

    return pd.DataFrame(rows)


strikes_by_maturity = {}
for T in true_params_by_maturity:
    F = forward_price(market_config.spot, T, market_config.risk_free_rate, market_config.dividend_yield)
    strikes_by_maturity[T] = np.round(F * np.array([0.70, 0.80, 0.90, 0.95, 1.00, 1.05, 1.10, 1.20, 1.30]), 2)

market_chain = generate_synthetic_sabr_chain(
    true_params_by_maturity,
    market_config,
    strikes_by_maturity,
    seed=market_config.seed
)

market_chain.head()

In [ ]:
plt.figure(figsize=(10, 6))
for T, group in market_chain.groupby("maturity"):
    group = group.sort_values("strike")
    plt.plot(group["strike"], group["market_iv"], marker="o", label=f"T={T:.2f}y")
plt.title("Synthetic SABR Market Smiles")
plt.xlabel("Strike")
plt.ylabel("Black-76 implied volatility")
plt.legend()
plt.show()

## 8. Calibration strategy

We fix:

$$
\beta=0.70
$$
and calibrate:

$$
(\alpha,\rho,\nu)
$$
for each maturity slice.

Why fix $\beta$?

- $\beta$ controls the backbone;
- one smile slice often cannot identify it robustly;
- calibrating $\beta$ freely can create unstable parameters;
- desks often choose $\beta$ by asset class or historical backbone analysis.

Parameter transforms:

$$
\alpha=e^{x_1}
$$
$$
\rho=\tanh(x_2)
$$
$$
\nu=e^{x_3}
$$

In [ ]:
def pack_sabr_slice(alpha, rho, nu):
    return np.array([np.log(alpha), np.arctanh(np.clip(rho, -0.999, 0.999)), np.log(nu)], dtype=float)


def unpack_sabr_slice(x, beta):
    return SABRParams(
        alpha=float(np.exp(x[0])),
        beta=float(beta),
        rho=float(np.tanh(x[1])),
        nu=float(np.exp(x[2]))
    )


def sabr_slice_residuals(x, slice_df, beta, use_price_residuals=False):
    params = unpack_sabr_slice(x, beta)
    residuals = []

    for row in slice_df.itertuples():
        model_iv = sabr_lognormal_vol(row.forward, row.strike, row.maturity, params)

        if use_price_residuals:
            model_price = black76_price(row.forward, row.strike, row.maturity, row.discount_factor, model_iv, "call")
            scale = max(row.bid_ask_spread, 0.02)
            residuals.append((model_price - row.mid) / scale)
        else:
            residuals.append((model_iv - row.market_iv) / 0.01)

    return np.array(residuals, dtype=float)

## 9. Calibrate one smile slice

We start with one maturity and inspect the result.

In [ ]:
fixed_beta = 0.70
sample_T = sorted(market_chain["maturity"].unique())[2]
sample_slice = market_chain[market_chain["maturity"] == sample_T].copy()

initial_alpha = sample_slice.loc[(sample_slice["log_moneyness"].abs()).idxmin(), "market_iv"] * sample_slice["forward"].iloc[0] ** (1 - fixed_beta)
initial_guess = pack_sabr_slice(alpha=initial_alpha, rho=-0.20, nu=0.30)

if SCIPY_AVAILABLE:
    result = least_squares(
        sabr_slice_residuals,
        x0=initial_guess,
        args=(sample_slice, fixed_beta),
        max_nfev=200
    )
    sample_calibrated = unpack_sabr_slice(result.x, fixed_beta)
    sample_status = {"success": bool(result.success), "cost": float(result.cost), "nfev": int(result.nfev), "message": result.message}
else:
    rng = np.random.default_rng(123)
    best_x = initial_guess
    best_score = np.inf
    for _ in range(400):
        candidate = initial_guess + rng.normal(0, [0.45, 0.90, 0.60])
        resid = sabr_slice_residuals(candidate, sample_slice, fixed_beta)
        score = float(np.mean(resid**2))
        if score < best_score:
            best_score = score
            best_x = candidate
    sample_calibrated = unpack_sabr_slice(best_x, fixed_beta)
    sample_status = {"success": True, "cost": best_score, "nfev": 400, "message": "SciPy unavailable; random-search fallback used."}

pd.Series({
    "sample_maturity": sample_T,
    **sample_status,
    **{f"calibrated_{k}": v for k, v in asdict(sample_calibrated).items()}
})

In [ ]:
def build_slice_fit(slice_df, params):
    rows = []
    for row in slice_df.itertuples():
        model_iv = sabr_lognormal_vol(row.forward, row.strike, row.maturity, params)
        model_price = black76_price(row.forward, row.strike, row.maturity, row.discount_factor, model_iv, "call")
        rows.append({
            "maturity": row.maturity,
            "forward": row.forward,
            "strike": row.strike,
            "log_moneyness": row.log_moneyness,
            "market_iv": row.market_iv,
            "model_iv": model_iv,
            "iv_error": model_iv - row.market_iv,
            "abs_iv_error": abs(model_iv - row.market_iv),
            "market_mid": row.mid,
            "model_price": model_price,
            "price_error": model_price - row.mid,
            "abs_price_error": abs(model_price - row.mid)
        })
    return pd.DataFrame(rows)


sample_fit = build_slice_fit(sample_slice, sample_calibrated)

sample_fit

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(sample_fit["strike"], sample_fit["market_iv"], marker="o", label="Market IV")
plt.plot(sample_fit["strike"], sample_fit["model_iv"], marker="x", label="Calibrated SABR IV")
plt.title(f"SABR Fit for Maturity T={sample_T:.2f}y")
plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.legend()
plt.show()

## 10. Calibrate every maturity slice

SABR is usually calibrated slice by slice.

This creates term structures of:

$$
\alpha(T),\quad \rho(T),\quad \nu(T)
$$
with fixed $\beta$.

In [ ]:
def calibrate_all_slices(chain, beta):
    param_rows = []
    fit_frames = []
    status_rows = []

    for T, slice_df in chain.groupby("maturity"):
        slice_df = slice_df.sort_values("strike").copy()
        F = slice_df["forward"].iloc[0]

        atm_row = slice_df.loc[(slice_df["log_moneyness"].abs()).idxmin()]
        alpha0 = max(atm_row["market_iv"] * F ** (1 - beta), 1e-4)
        x0 = pack_sabr_slice(alpha=alpha0, rho=-0.20, nu=0.30)

        if SCIPY_AVAILABLE:
            res = least_squares(
                sabr_slice_residuals,
                x0=x0,
                args=(slice_df, beta),
                max_nfev=250
            )
            p = unpack_sabr_slice(res.x, beta)
            status = {"success": bool(res.success), "cost": float(res.cost), "nfev": int(res.nfev), "message": res.message}
        else:
            rng = np.random.default_rng(int(T * 100000))
            best_x = x0
            best_score = np.inf
            for _ in range(500):
                candidate = x0 + rng.normal(0, [0.45, 0.85, 0.60])
                score = float(np.mean(sabr_slice_residuals(candidate, slice_df, beta)**2))
                if score < best_score:
                    best_score = score
                    best_x = candidate
            p = unpack_sabr_slice(best_x, beta)
            status = {"success": True, "cost": best_score, "nfev": 500, "message": "random-search fallback"}

        fit_df = build_slice_fit(slice_df, p)
        fit_frames.append(fit_df)

        true_row = slice_df.iloc[0]

        param_rows.append({
            "maturity": T,
            "calibrated_alpha": p.alpha,
            "calibrated_beta": p.beta,
            "calibrated_rho": p.rho,
            "calibrated_nu": p.nu,
            "true_alpha": true_row["true_alpha"],
            "true_beta": true_row["true_beta"],
            "true_rho": true_row["true_rho"],
            "true_nu": true_row["true_nu"]
        })

        status_rows.append({
            "maturity": T,
            **status
        })

    params_df = pd.DataFrame(param_rows).sort_values("maturity")
    fit_report = pd.concat(fit_frames, ignore_index=True)
    status_df = pd.DataFrame(status_rows).sort_values("maturity")

    return params_df, fit_report, status_df


calibrated_params_by_maturity, fit_report, calibration_status = calibrate_all_slices(
    market_chain,
    beta=fixed_beta
)

calibrated_params_by_maturity

In [ ]:
calibration_status

## 11. Fit diagnostics

A useful SABR calibration report should include:

- IV RMSE;
- IV MAE;
- price RMSE;
- residuals by strike and maturity;
- parameter term structures.

In [ ]:
fit_summary = (
    fit_report
    .groupby("maturity")
    .agg(
        n_quotes=("market_iv", "count"),
        iv_rmse=("iv_error", lambda x: float(np.sqrt(np.mean(x**2)))),
        iv_mae=("abs_iv_error", "mean"),
        max_abs_iv_error=("abs_iv_error", "max"),
        price_rmse=("price_error", lambda x: float(np.sqrt(np.mean(x**2)))),
        price_mae=("abs_price_error", "mean")
    )
    .reset_index()
)

fit_summary

In [ ]:
overall_fit_summary = pd.Series({
    "overall_iv_rmse": float(np.sqrt(np.mean(fit_report["iv_error"]**2))),
    "overall_iv_mae": float(fit_report["abs_iv_error"].mean()),
    "overall_max_abs_iv_error": float(fit_report["abs_iv_error"].max()),
    "overall_price_rmse": float(np.sqrt(np.mean(fit_report["price_error"]**2))),
    "overall_price_mae": float(fit_report["abs_price_error"].mean())
})

overall_fit_summary

In [ ]:
plt.figure(figsize=(11, 6))

for T, group in fit_report.groupby("maturity"):
    group = group.sort_values("strike")
    plt.plot(group["strike"], group["market_iv"], marker="o", linestyle="--", alpha=0.65, label=f"Market T={T:.2f}")
    plt.plot(group["strike"], group["model_iv"], marker="x", alpha=0.65, label=f"SABR T={T:.2f}")

plt.title("Market vs Calibrated SABR Smiles")
plt.xlabel("Strike")
plt.ylabel("Black-76 implied volatility")
plt.legend(ncol=2, fontsize=8)
plt.show()

In [ ]:
residual_pivot = fit_report.pivot(index="maturity", columns="strike", values="iv_error")

plt.figure(figsize=(11, 5))
plt.imshow(
    residual_pivot.values,
    origin="lower",
    aspect="auto",
    extent=[
        residual_pivot.columns.min(),
        residual_pivot.columns.max(),
        residual_pivot.index.min(),
        residual_pivot.index.max()
    ]
)
plt.colorbar(label="Model IV - Market IV")
plt.title("SABR IV Residual Heatmap")
plt.xlabel("Strike")
plt.ylabel("Maturity")
plt.show()

## 12. Parameter term structures

SABR calibration gives maturity-dependent parameters.

A smooth and plausible parameter term structure is often as important as a low one-day error.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(calibrated_params_by_maturity["maturity"], calibrated_params_by_maturity["true_alpha"], marker="o", linestyle="--", label="true alpha")
plt.plot(calibrated_params_by_maturity["maturity"], calibrated_params_by_maturity["calibrated_alpha"], marker="x", label="calibrated alpha")
plt.title("SABR Alpha Term Structure")
plt.xlabel("Maturity")
plt.ylabel("Alpha")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(calibrated_params_by_maturity["maturity"], calibrated_params_by_maturity["true_rho"], marker="o", linestyle="--", label="true rho")
plt.plot(calibrated_params_by_maturity["maturity"], calibrated_params_by_maturity["calibrated_rho"], marker="x", label="calibrated rho")
plt.title("SABR Rho Term Structure")
plt.xlabel("Maturity")
plt.ylabel("Rho")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(calibrated_params_by_maturity["maturity"], calibrated_params_by_maturity["true_nu"], marker="o", linestyle="--", label="true nu")
plt.plot(calibrated_params_by_maturity["maturity"], calibrated_params_by_maturity["calibrated_nu"], marker="x", label="calibrated nu")
plt.title("SABR Nu Term Structure")
plt.xlabel("Maturity")
plt.ylabel("Nu")
plt.legend()
plt.show()

## 13. Smile feature interpretation

For each maturity, we extract simple smile features:

1. ATM volatility;
2. left-right skew proxy;
3. curvature proxy.

These features help connect SABR parameters to surface research.

In [ ]:
def sabr_smile_features(params_by_T, market: SABRMarketConfig):
    rows = []

    for row in params_by_T.itertuples():
        T = float(row.maturity)
        params = SABRParams(
            alpha=float(row.calibrated_alpha),
            beta=float(row.calibrated_beta),
            rho=float(row.calibrated_rho),
            nu=float(row.calibrated_nu)
        )
        F = forward_price(market.spot, T, market.risk_free_rate, market.dividend_yield)

        k_values = [-0.20, -0.10, 0.0, 0.10, 0.20]
        vols = {}
        for k in k_values:
            K = F * exp(k)
            vols[k] = sabr_lognormal_vol(F, K, T, params)

        rows.append({
            "maturity": T,
            "atm_iv": vols[0.0],
            "skew_10_logmoneyness": vols[-0.10] - vols[0.10],
            "curvature_10_logmoneyness": 0.5 * (vols[-0.10] + vols[0.10]) - vols[0.0],
            "left_wing_minus_atm": vols[-0.20] - vols[0.0],
            "right_wing_minus_atm": vols[0.20] - vols[0.0]
        })

    return pd.DataFrame(rows)


smile_features = sabr_smile_features(calibrated_params_by_maturity, market_config)

smile_features

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(smile_features["maturity"], smile_features["atm_iv"], marker="o", label="ATM IV")
plt.plot(smile_features["maturity"], smile_features["skew_10_logmoneyness"], marker="o", label="Skew proxy")
plt.plot(smile_features["maturity"], smile_features["curvature_10_logmoneyness"], marker="o", label="Curvature proxy")
plt.title("SABR-Derived Smile Feature Term Structures")
plt.xlabel("Maturity")
plt.ylabel("Feature value")
plt.legend()
plt.show()

## 14. Sensitivity to $\rho$, $\nu$, and $\beta$

SABR parameters have clear qualitative roles:

- $\rho$: skew direction and steepness;
- $\nu$: curvature and wing steepness;
- $\beta$: backbone relationship between forward and volatility.

We perturb each parameter around a calibrated one-year slice.

In [ ]:
one_year_row = calibrated_params_by_maturity.iloc[(calibrated_params_by_maturity["maturity"] - 1.0).abs().argmin()]
T_sens = float(one_year_row["maturity"])
F_sens = forward_price(market_config.spot, T_sens, market_config.risk_free_rate, market_config.dividend_yield)
K_grid = F_sens * np.exp(np.linspace(-0.35, 0.35, 41))

base_sabr = SABRParams(
    alpha=float(one_year_row.calibrated_alpha),
    beta=float(one_year_row.calibrated_beta),
    rho=float(one_year_row.calibrated_rho),
    nu=float(one_year_row.calibrated_nu)
)


def sensitivity_smile(base, param_name, values):
    rows = []
    for val in values:
        p_dict = asdict(base)
        p_dict[param_name] = float(val)
        p = SABRParams(**p_dict)
        for K in K_grid:
            rows.append({
                "parameter": param_name,
                "parameter_value": float(val),
                "strike": float(K),
                "log_moneyness": log(float(K) / F_sens),
                "iv": sabr_lognormal_vol(F_sens, float(K), T_sens, p)
            })
    return pd.DataFrame(rows)


rho_sens = sensitivity_smile(base_sabr, "rho", [-0.80, -0.50, -0.20, 0.20])
nu_sens = sensitivity_smile(base_sabr, "nu", [0.15, 0.30, 0.55, 0.80])
beta_sens = sensitivity_smile(base_sabr, "beta", [0.30, 0.50, 0.70, 0.90])

In [ ]:
plt.figure(figsize=(10, 6))
for val, group in rho_sens.groupby("parameter_value"):
    plt.plot(group["strike"], group["iv"], label=f"rho={val}")
plt.title("SABR Smile Sensitivity to rho")
plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
for val, group in nu_sens.groupby("parameter_value"):
    plt.plot(group["strike"], group["iv"], label=f"nu={val}")
plt.title("SABR Smile Sensitivity to nu")
plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
for val, group in beta_sens.groupby("parameter_value"):
    plt.plot(group["strike"], group["iv"], label=f"beta={val}")
plt.title("SABR Smile Sensitivity to beta")
plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.legend()
plt.show()

## 15. Fixed beta versus free beta

It is tempting to calibrate $\beta$ freely.

But one maturity slice often cannot reliably identify all four parameters:

$$
(\alpha,\beta,\rho,\nu)
$$
Problems include:

- parameter instability;
- multiple local minima;
- unrealistic backbone;
- overfitting smile noise;
- poor extrapolation.

Practical approaches:

1. fix $\beta$ by asset class;
2. estimate $\beta$ from historical backbone behaviour;
3. calibrate $\beta$ globally across maturities;
4. impose regularisation across maturities.

## 16. Shifted SABR note

The lognormal SABR formula requires positive forwards and strikes.

For low or negative rates, practitioners often use shifted SABR:

$$
F^{shift}=F+s
$$
$$
K^{shift}=K+s
$$
where $s$ is a positive shift chosen so that shifted forwards and strikes are positive.

This notebook does not implement shifted SABR, but the engineering change is conceptually simple: apply the SABR formula to shifted $F$ and $K$, then price with the corresponding shifted convention.

For commodity and futures options with positive forwards, unshifted lognormal SABR is often sufficient as a first model.

## 17. Calibration hygiene checklist

A SABR calibration should report:

1. fixed or calibrated $\beta$;
2. optimisation status;
3. parameter constraints;
4. IV RMSE and MAE;
5. residuals by strike;
6. residuals by maturity;
7. parameter term structures;
8. sensitivity to $\rho,\nu,\beta$;
9. extrapolation behaviour;
10. whether the model is lognormal, normal, or shifted.

## 18. Saving outputs

The notebook saves:

1. synthetic SABR option chain;
2. calibrated parameters by maturity;
3. calibration status;
4. fit report;
5. fit summary;
6. smile features;
7. sensitivity tables;
8. manifest.

In [ ]:
output_dir = Path("data/processed/sabr_smile_calibration_v1")
output_dir.mkdir(parents=True, exist_ok=True)

market_chain_path = output_dir / "synthetic_sabr_market_chain.csv"
params_path = output_dir / "calibrated_sabr_params_by_maturity.csv"
status_path = output_dir / "calibration_status.csv"
fit_report_path = output_dir / "fit_report.csv"
fit_summary_path = output_dir / "fit_summary.csv"
overall_summary_path = output_dir / "overall_fit_summary.csv"
features_path = output_dir / "smile_features.csv"
rho_sens_path = output_dir / "rho_sensitivity.csv"
nu_sens_path = output_dir / "nu_sensitivity.csv"
beta_sens_path = output_dir / "beta_sensitivity.csv"
manifest_path = output_dir / "manifest.json"

market_chain.to_csv(market_chain_path, index=False)
calibrated_params_by_maturity.to_csv(params_path, index=False)
calibration_status.to_csv(status_path, index=False)
fit_report.to_csv(fit_report_path, index=False)
fit_summary.to_csv(fit_summary_path, index=False)
overall_fit_summary.to_frame("value").to_csv(overall_summary_path)
smile_features.to_csv(features_path, index=False)
rho_sens.to_csv(rho_sens_path, index=False)
nu_sens.to_csv(nu_sens_path, index=False)
beta_sens.to_csv(beta_sens_path, index=False)

manifest = {
    "dataset_name": "sabr_smile_calibration_outputs",
    "schema_version": "sabr_smile_calibration_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "market_config": asdict(market_config),
    "fixed_beta": fixed_beta,
    "true_params_by_maturity": {
        str(T): asdict(p) for T, p in true_params_by_maturity.items()
    },
    "overall_fit_summary": overall_fit_summary.to_dict(),
    "scipy_available": SCIPY_AVAILABLE,
    "notes": [
        "Synthetic market IVs are generated from Hagan lognormal SABR smiles.",
        "Calibration fixes beta and fits alpha, rho, and nu per maturity.",
        "Objective is based on IV residuals scaled by 1 vol point.",
        "Black-76 is used because SABR is naturally forward based.",
        "Shifted SABR is discussed but not implemented."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

market_chain_path, params_path, fit_report_path, manifest_path

## 19. Limitations

### 19.1 Hagan approximation

The Hagan formula is an approximation, not the exact SABR price.

Accuracy can degrade for extreme strikes, long maturities, or extreme parameters.

### 19.2 Fixed beta

Fixing $\beta$ improves stability but may impose the wrong backbone.

Free $\beta$ calibration can overfit.

### 19.3 Slice-by-slice calibration

Each maturity is calibrated independently.

This can create noisy parameter term structures unless regularised.

### 19.4 No arbitrage not guaranteed

Raw SABR slices do not automatically guarantee a globally arbitrage-free volatility surface.

Calendar and butterfly arbitrage checks are still required.

### 19.5 Positive forward assumption

The lognormal SABR formula assumes positive forwards and strikes.

Rates markets often require shifted or normal SABR.

### 19.6 Synthetic data

Real data needs quote cleaning, bid/ask handling, forward construction, holiday calendars, stale quote filtering, and surface validation.

## 20. Research frontier and extensions

Important extensions include:

1. **normal SABR** for low/negative rates;
2. **shifted SABR** for positive-shift modelling;
3. **arbitrage-free SABR interpolation**;
4. **global beta calibration** across maturities;
5. **regularised parameter term structures**;
6. **SABR Greeks and risk reports**;
7. **SABR vs SVI comparison**;
8. **SABR calibration for futures options**;
9. **neural calibration surrogates** for fast surface fitting.

For this repo, a natural extension is a Chinese futures options notebook where forwards, contract calendars, tick sizes, and product-specific liquidity matter.

## 21. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_14_local_volatility_dupire_surface**  
   Surface-to-local-volatility construction.

2. **02_15_jump_diffusion_pide_pricer**  
   PIDE pricing with jumps.

3. **03_09_volatility_surface_alpha_signals**  
   Turning SABR features into alpha/risk signals.

4. **04_07_beta_weighted_tail_risk_hedging**  
   Using skew and curvature for hedging decisions.

5. **07_02_volatility_surface_pricing_and_alpha**  
   Capstone combining surface construction, SABR/Heston calibration, and volatility alpha.

6. **Chinese futures options extension**  
   Using Black-76/SABR on commodity futures option smiles.

## 22. Summary

This notebook implemented SABR smile calibration.

It showed how to:

1. define SABR forward dynamics;
2. implement Hagan lognormal implied-volatility approximation;
3. use Black-76 pricing for forward options;
4. generate synthetic SABR smiles;
5. calibrate $\alpha,\rho,\nu$ with fixed $\beta$;
6. compare true and calibrated parameters;
7. diagnose IV and price residuals;
8. extract smile features;
9. study sensitivity to $\rho,\nu,\beta$;
10. document shifted SABR and calibration limitations.

The key computational takeaway:

> SABR calibration is a constrained nonlinear optimisation problem wrapped around an implied-volatility approximation.

The key financial takeaway:

> SABR gives interpretable smile parameters, but those parameters must be stabilised, diagnosed, and treated as a surface model rather than unquestioned truth.

## 23. Further reading

- Hagan, Kumar, Lesniewski, and Woodward, "Managing Smile Risk."
- Rebonato, *Volatility and Correlation*.
- Gatheral, *The Volatility Surface*.
- Andersen and Piterbarg, volatility modelling references.
- Obłój, "Fine-tune your smile: correction to Hagan et al."
- Literature on shifted SABR and normal SABR.
- Literature on arbitrage-free smile interpolation.